# Momentum — Execution Scenario Analysis
Tests how portfolio OOS performance changes depending on when trades are filled.
Signals and trade logic are **never modified** — only the fill price changes.

Hourly data is fetched **once** — swap execution time in cell 2 without re-fetching.

---
### How the return engine works
`plot_portfolio_oos._strat_ret` computes per bar:

```
ret[T] = position[T-1] × pct_change(Close)[T] × position_size[T]
```

The **first-entry bar T** always contributes **zero** (position[T-1] = 0).  
The first *captured* return is bar **T+1**: `(Close[T+1] - Close[T]) / Close[T]`.

---
### How scenario fills are applied

Two bars are modified per trade:

| Bar | Condition | What we set | Resulting captured return |
|---|---|---|---|
| **First-entry T** | pos[T]≠0, pos[T-1]=0 | `Close[T] = 10am_T` | `ret[T+1] = (Close[T+1] − 10am_T) / 10am_T` |
| **Last-active T'** | pos[T']≠0, pos[T'+1]=0 | `Close[T'] = 10am_{T'+1}` | `ret[T'] = (10am_{T'+1} − Close[T'−1]) / Close[T'−1]` |

Middle bars are untouched (close-to-close). **Total trade return = 10am_T → 10am_{T'+1}.**

---
### Scenarios

| Scenario | Entry fills at | Exit fills at |
|---|---|---|
| `'close'` | Close[T] midnight UTC — baseline (no modification) | same |
| `integer` | HH:00 UTC on entry day T | HH:00 UTC on day after exit signal T'+1 |
| `'worst_long'` | High[T+1] — worst intraday buy | Low[T'+1] — worst intraday sell |


In [80]:
import os
import sys
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ─────────────────────────────────
ROOT   = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows

WF_DIR = os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'wf_testing')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos

client = get_binance_client()

---
## 1 — Load OOS results and fetch hourly data (run once)

In [70]:
# ── load daily OOS pkl files ───────────────────────────────────────────────────
coin_dfs = {}
for fname in os.listdir(WF_DIR):
    if fname.endswith('_oos.pkl'):
        coin = fname.replace('_oos.pkl', '').upper()
        coin_dfs[coin] = pd.read_pickle(os.path.join(WF_DIR, fname))

print(f'Loaded daily OOS: {list(coin_dfs.keys())}')

# ── fetch 1h data for every coin (one-time pull) ───────────────────────────────
hourly = {}   # coin → 1h DataFrame

for coin, df in coin_dfs.items():
    symbol = coin + 'USDT'
    start  = str(df.index[0].date())
    end    = str(df.index[-1].date())
    klines = client.get_historical_klines(symbol, '1h', start, end)

    h = pd.DataFrame(klines, columns=[
        'Time','Open','High','Low','Close','Volume',
        'Close_time','Quote_volume','Trades','Taker_base','Taker_quote','Ignore'
    ])
    h['Time'] = pd.to_datetime(h['Time'], unit='ms', utc=True)
    for col in ['Open','High','Low','Close']:
        h[col] = h[col].astype(float)
    h = h.set_index('Time')
    hourly[coin] = h
    print(f'  {coin}: {len(h)} hourly bars  ({start} → {end})')

print('\nHourly data ready.')

Loaded daily OOS: ['ADA', 'XRP', 'ETH', 'BTC', 'SOL', 'BNB', 'AVAX']


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_6075/2174840675.py:17: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  ADA: 26281 hourly bars  (2023-04-06 → 2026-04-05)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_6075/2174840675.py:17: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  XRP: 26377 hourly bars  (2023-04-01 → 2026-04-04)


...

  SOL: 23017 hourly bars  (2023-06-27 → 2026-02-10)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_6075/2174840675.py:17: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  BNB: 26281 hourly bars  (2023-03-31 → 2026-03-30)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_6075/2174840675.py:17: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  AVAX: 22993 hourly bars  (2023-08-08 → 2026-03-23)

Hourly data ready.


---
## 2 — Set execution scenario (re-run this cell to switch, no re-fetch needed)

**Options:**
- `SCENARIO = 'close'` — baseline: midnight UTC Close on signal day
- `SCENARIO = 10` — 10am UTC on the day after the signal (any int 0–23)
- `SCENARIO = 'worst_long'` — enter at High[T+1], exit at Low[T+1]

In [117]:
SCENARIO = 9# ← 'close'  |  integer hour 0-23

def apply_scenario(coin_dfs, hourly, scenario):
    if scenario == 'close':
        return coin_dfs

    adjusted = {}
    for coin, df in coin_dfs.items():
        d = df.copy()
        h = hourly[coin].copy()

        # 1. Get the price at the specific hour
        prices_hh = h.loc[h.index.hour == scenario, 'Open'].copy()

        # 2. CRITICAL ALIGNMENT: 
        # If scenario is 0 (00:00), this price is effectively the "Close" 
        # of the PREVIOUS day. We must shift the index back by 1 day 
        # so it aligns with the daily row that generated the signal.
        if isinstance(scenario, int):
            # Move the price of 'Tuesday 00:00' to the 'Monday' row
            prices_hh.index = prices_hh.index.tz_convert(None) - pd.Timedelta(days=1)
            prices_hh.index = prices_hh.index.normalize()

        # 3. Map to the daily dataframe
        prices_hh = prices_hh[~prices_hh.index.duplicated(keep='first')]
        d['Close'] = prices_hh.reindex(d.index).ffill()

        # Remove the 'synthetic_close' / cumprod logic entirely.
        # This prevents "double-calculating" returns if your 
        # downstream code also calculates returns.
        
        adjusted[coin] = d

    return adjusted

coin_dfs_exec = apply_scenario(coin_dfs, hourly, SCENARIO)
label = f'{SCENARIO}h_UTC' if isinstance(SCENARIO, int) else SCENARIO
print(f'\nScenario applied: {label}')




Scenario applied: 9h_UTC


---
## 3 — Portfolio OOS performance

In [ ]:
SHOW_COINS = ['ETH', 'XRP', 'AVAX', 'SOL', 'BNB']   # or list(coin_dfs.keys())

metrics = plot_portfolio_oos(
    coin_dfs   = coin_dfs_exec,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},   # benchmark always uses original close
    show       = True,
    save_html  = os.path.join(
        ROOT, 'topics', 'momentum', 'results',
        f'portfolio_{label}.html'
    ),
)

# ── portfolio summary ─────────────────────────────────────────────────────────
print(f'\n── Portfolio  |  Scenario: {label} ──')
print(f'  {"Total Return":<22} {metrics["total_return"]*100:.2f}%')
print(f'  {"Sharpe Ratio":<22} {metrics["sharpe_ratio"]:.2f}')
print(f'  {"Max Drawdown":<22} {metrics["max_drawdown"]*100:.2f}%')
print(f'  {"Calmar Ratio":<22} {metrics["calmar_ratio"]:.2f}')
print(f'  {"Total Trades":<22} {metrics["num_trades"]}')
print(f'  {"Win Rate":<22} {metrics["win_rate"]*100:.2f}%')
print(f'  {"Profit Factor":<22} {metrics["profit_factor"]:.2f}')
print(f'  {"Avg Win/Loss":<22} {metrics["avg_win_loss_ratio"]:.2f}')

# ── per-coin trade statistics table ──────────────────────────────────────────
print(f'\n── Per-Coin Trade Statistics  |  Scenario: {label} ──')
print(f'  {"Coin":<6} {"Trades":>7} {"Win Rate":>10} {"Prof.Factor":>13} {"Avg W/L":>9}')
print(f'  {"─"*6} {"─"*7} {"─"*10} {"─"*13} {"─"*9}')

for coin in SHOW_COINS:
    t = metrics['coin_trade_stats'].get(coin, None)
    if t is None or len(t) == 0:
        print(f'  {coin:<6} {"0":>7} {"—":>10} {"—":>13} {"—":>9}')
        continue
    n      = len(t)
    wins   = t[t['pnl'] > 0]
    losses = t[t['pnl'] < 0]
    wr     = len(wins) / n if n else 0.0
    gp     = wins['pnl'].sum()
    gl     = abs(losses['pnl'].sum())
    pf     = gp / gl if gl > 0 else 0.0
    aw     = wins['pnl'].mean()        if len(wins)   > 0 else 0.0
    al     = abs(losses['pnl'].mean()) if len(losses) > 0 else 0.0
    awl    = aw / al                   if al          > 0 else 0.0
    print(f'  {coin:<6} {n:>7} {wr*100:>9.1f}% {pf:>13.2f} {awl:>9.2f}')